# ISOM 835 · Session 5 — Classification: Predicting Yes/No
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Oct 19 · Prof. Hasan Arslan**

Logistic regression as odds, kNN and Naive Bayes as contrasts, the confusion matrix cell by cell, precision vs. recall, ROC vs. PR — on 30,000 credit-card customers, 22% of whom defaulted.

> **Frame the prediction.** *Unit:* one cardholder · *Target:* default next month (1/0) · *Horizon:* one month · *Decision:* cut the credit line / call / nothing · *Baseline:* "nobody defaults" = 77.9% accuracy.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

credit = fetch_openml(data_id=42477, as_frame=True)
names = ['limit_bal', 'sex', 'education', 'marriage', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6',
         'bill_1', 'bill_2', 'bill_3', 'bill_4', 'bill_5', 'bill_6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6']
X = credit.data.set_axis(names, axis=1).astype(float)
y = (credit.target == '1').astype(int)
print(X.shape, f'default rate {y.mean():.1%}')
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)

## 1. Why not linear regression on a 0/1 target?
A line predicts −0.3 and 1.4 — not probabilities. The **sigmoid** squashes any score into (0, 1), and logistic regression fits a *line in log-odds*: log(p / (1−p)) = β₀ + β₁x₁ + …

In [ ]:
z = np.linspace(-6, 6, 200)
plt.figure(figsize=(6, 3)); plt.plot(z, 1 / (1 + np.exp(-z)), color='#2ee6c5', lw=2); plt.axhline(0.5, ls='--', color='#f5a524'); plt.xlabel('linear score z'); plt.ylabel('P(default)'); plt.title('sigmoid: z → probability'); plt.show()

In [ ]:
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X_tr, y_tr)
p_logit = logit.predict_proba(X_te)[:, 1]
print(f'logistic  ROC-AUC {roc_auc_score(y_te, p_logit):.3f}   PR-AUC {average_precision_score(y_te, p_logit):.3f}')

## 2. Read the coefficients as odds ratios
On standardized features, exp(β) is the multiplicative change in the *odds* of default per one standard deviation. The payment-status columns (`pay_0` = last month's repayment status) dominate.

In [ ]:
coef = pd.Series(logit.named_steps['logisticregression'].coef_[0], index=X.columns)
odds = np.exp(coef).sort_values(ascending=False)
print('odds ratio per +1 SD (top 5 / bottom 3):'); print(odds.head(5).round(2)); print(odds.tail(3).round(2))
print(f"\nA one-SD increase in pay_0 (being later on last month's payment) multiplies the odds of default by {odds['pay_0']:.2f}.")

## 3. Two other ways to draw a boundary
**kNN**: no training, just a vote among the k nearest customers — a boundary that follows the data everywhere (and needs scaling). **Gaussian Naive Bayes**: multiply per-feature likelihoods assuming independence — fast, surprisingly decent, badly calibrated.

In [ ]:
models = {'logistic': logit,
          'kNN (k=25)': make_pipeline(StandardScaler(), KNeighborsClassifier(25)).fit(X_tr, y_tr),
          'naive bayes': make_pipeline(StandardScaler(), GaussianNB()).fit(X_tr, y_tr)}
probs = {n: m.predict_proba(X_te)[:, 1] for n, m in models.items()}
for n, p in probs.items():
    print(f'{n:12s} ROC-AUC {roc_auc_score(y_te, p):.3f}   PR-AUC {average_precision_score(y_te, p):.3f}')

In [ ]:
# decision boundaries on two features, to see the three personalities
from matplotlib.colors import ListedColormap
f2 = ['pay_0', 'limit_bal']; sub = X_tr.sample(3000, random_state=835); ysub = y_tr.loc[sub.index]
xx, yy = np.meshgrid(np.linspace(-2, 8, 120), np.linspace(0, 8e5, 120))
grid = pd.DataFrame({'pay_0': xx.ravel(), 'limit_bal': yy.ravel()})
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (n, cls) in zip(axes, [('logistic', LogisticRegression(max_iter=2000)), ('kNN (k=25)', KNeighborsClassifier(25)), ('naive bayes', GaussianNB())]):
    m = make_pipeline(StandardScaler(), cls).fit(sub[f2], ysub)
    ax.contourf(xx, yy, m.predict_proba(grid)[:, 1].reshape(xx.shape), levels=20, cmap='RdYlGn_r', alpha=0.6)
    ax.scatter(sub['pay_0'] + np.random.default_rng(1).normal(0, 0.08, len(sub)), sub['limit_bal'], c=ysub, cmap=ListedColormap(['#2ee6c5', '#ff6b8b']), s=4, alpha=0.5)
    ax.set_title(n); ax.set_xlabel('pay_0 (months late)'); ax.set_ylabel('credit limit')
plt.tight_layout(); plt.show()

## 4. The confusion matrix, cell by cell
`predict()` applies a 0.5 cutoff you never chose. Read the four cells in the bank's language:

| | predicted default | predicted OK |
|---|---|---|
| **actually defaults** | TP — caught | **FN — approved, then lost** (the expensive cell) |
| **actually OK** | **FP — good customer declined** (lost revenue, lost trust) | TN — correctly left alone |

In [ ]:
pred = logit.predict(X_te)
cm = confusion_matrix(y_te, pred); tn, fp, fn, tp = cm.ravel()
print(cm)
print(f'precision = TP/(TP+FP) = {tp}/{tp+fp} = {tp/(tp+fp):.3f}')
print(f'recall    = TP/(TP+FN) = {tp}/{tp+fn} = {tp/(tp+fn):.3f}')
print(f'accuracy  = (TP+TN)/N  = {(tp+tn)/len(y_te):.3f}   vs. baseline {1-y_te.mean():.3f}')
print(classification_report(y_te, pred, target_names=['no default', 'default']))
ConfusionMatrixDisplay(cm, display_labels=['no default', 'default']).plot(cmap='Blues'); plt.show()

Accuracy 81% vs. baseline 78% — and the model catches fewer than a quarter of the defaulters. That is not a modeling failure; it is a *decision* nobody made. Next week we price the two mistakes.

## 5. ROC vs. precision–recall
ROC sweeps every threshold and plots recall vs. false-positive rate; **AUC = P(random defaulter scores above random non-defaulter)**. PR plots precision vs. recall. At a 22% base rate both are informative; at a 0.2% fraud rate only PR tells the truth.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n, p in probs.items():
    RocCurveDisplay.from_predictions(y_te, p, name=n, ax=ax[0]); PrecisionRecallDisplay.from_predictions(y_te, p, name=n, ax=ax[1])
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[1].axhline(y_te.mean(), ls='--', color='gray', label=f'base rate {y_te.mean():.2f}'); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Class imbalance — the fixes, in order
1. **Pick the right metric** (PR-AUC, recall at a fixed precision). 2. **Reweight**: `class_weight='balanced'`. 3. **Tune the threshold** (Session 6). 4. Only then consider resampling (SMOTE) — with modern models it rarely helps and can distort calibration.

In [ ]:
bal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced')).fit(X_tr, y_tr)
p_bal = bal.predict_proba(X_te)[:, 1]
print(f'balanced logistic: ROC-AUC {roc_auc_score(y_te, p_bal):.3f}  PR-AUC {average_precision_score(y_te, p_bal):.3f}')
print('confusion at 0.5 (balanced weights):'); print(confusion_matrix(y_te, bal.predict(X_te)))
print('→ same ranking quality, very different confusion matrix: reweighting just moved the cutoff in disguise')

## 7. Payment status as categories (a feature-engineering win)
`pay_0…pay_6` are codes (−2, −1, 0, 1, 2, …). Treating them as categories instead of numbers lets logistic regression capture the jump at "one month late".

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
pay = ['pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']; rest = [c for c in X.columns if c not in pay]
prep = ColumnTransformer([('num', StandardScaler(), rest), ('pay', OneHotEncoder(handle_unknown='ignore'), pay)])
cat_logit = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=5000))]).fit(X_tr, y_tr)
p_cat = cat_logit.predict_proba(X_te)[:, 1]
print(f'payment codes as categories: ROC-AUC {roc_auc_score(y_te, p_cat):.3f}  PR-AUC {average_precision_score(y_te, p_cat):.3f}')

## 8. Your turn
1. **Threshold preview.** Recompute the confusion matrix for the logistic model at thresholds 0.3 and 0.2. How do precision and recall trade off?
2. **k sweep.** For kNN, try k = 1, 5, 25, 101, 401 and report *both* training and test ROC-AUC. Where does the train–test gap close, where does test AUC peak, and what does that say about bias–variance?
3. **Odds in words.** Pick two odds ratios from Section 2 and write each as a sentence a credit officer would understand.

In [ ]:
# Your turn — work here

## What we learned tonight
- Logistic regression is **linear in log-odds**; exponentiate a coefficient to get an odds ratio and say it in words.
- The confusion matrix has **four cells with four business meanings**; precision and recall ask different questions; accuracy hides which mistake you are making.
- **ROC-AUC measures ranking; PR-AUC measures usefulness on rare positives.** `predict()`'s 0.5 is a decision you have not yet made.

**Homework #3** (due Mon Nov 2): credit-default classifier + the threshold/cost part after Session 6. Read ISLP 4.1–4.4.